In [80]:
#room specific code only activates when using parser
#Try to make keys easier to make.
#try to make [room].addDescription(n-, "description") mean that after n times, that becomes the default text, unless there is a description located past it.
#make additional interactions for objects work with more than one word.
#can't talk to npcs unless they enter the room at the same time, so you don't see them.
#add the ability to generate random rooms
#add the ability to add random maps.
#add [room].addUnderstanding() which is a room equivalent of [player].addUnderstanding().
#add contextual dialogue to NPCS
#ponder if Player.inventory could be better as a dictionary
#automatically list directions the player can move, unles specified otherwise.
#if I make conversations part of the actual gameplay, then I'd have to record them in the save file.
#make NPC.name, Room.name, and Thing.name be lists which allow for synonyms.
#I don't know how to improve this, but the dialogue replacement code seems inefficient
from just_playback import Playback
import climage, os, jaraco, appdirs, sys, random
class NPC:
    def __init__(self, name, description, **kwargs):
        self.name = name
        self.description = description
        self.dialogue = {}
        self.placeInPath = 0
        self.properties = []
        self.pathingDirs = []
        self.location = Room
        self.listed = True
        #changes properties if listed in arguments, otherwise creates new property
        for item in kwargs:
            if("listed" in item):   
                self.listed = kwargs["listed"]
            else:
                self.properties.append(item)
    def addDialogue(self, num, key, text, persistent=False):
        self.dialogue[str(num)] = [key, text, persistent]
    def replaceDialogue(self,depth, num, key, text,persistent=False):
        self.dialogue[str(num)+".N"*depth] = [key, text, persistent]
    def getDialogue(self, num):
        for key in self.dialogue.keys():
            if num == key:
                return f"\nThey communicate: {self.dialogue[str(num)][1]}"
    def remDialogue(self, num):
        to_delete = None
        if num in self.dialogue.keys() and self.dialogue[num][2]!=True:
            del self.dialogue[str(num)]
            for key in list(self.dialogue.keys()):
                #starts the resurrection.
                if num in key and ".N" in key:
                    self.dialogue[key[0:-2]] = self.dialogue[key]
            
    def setupPath(self, pathingDirs, location):
        """Makes the path, if there is an xn at the end, that part is multiplied n times. If the direction is empty, then the npc stays."""
        newList = []
        self.location = location
        if len(pathingDirs) == 1:
            #If the length is only 1 item
            if "x" in pathingDirs:
                #Checks if you are multiplying the specified direction
                multiplier = pathingDirs.split("x")[1]
                path = pathingDirs.split("x")[0]
                for item in range(int(number)):
                    newList.append(path)
                self.pathingDirs = newList 
                return
            else:
                self.pathingDirs = pathingDirs
                return
        #If the length is greater than 1 item.
        for item in pathingDirs:
            if("X" in item):
                multiplier = item.split("x")[1]
                path = item.split("x")[0]
                #Checks if you are multiplying the specified direction
                for item in range(int(multiplier)):
                    newList.append(path)
            else:
                newList.append(item)
        self.pathingDirs = newList
    def moveInPath(self, npc):
        """Progresses the NPC forward in their path, if they have one"""
        position =self.placeInPath % len(self.pathingDirs)
        for room, direction in self.location.exits:
            if direction == self.pathingDirs[position]:
                self.location.things.remove(self)
                room.things.append(npc)
                self.location = room
                self.placeInPath+=1
                return
    def talk(self):
        """A loop that allows for you to talk to this NPC without using the parser."""
        if not self.dialogue.items():
            return
        while True:
            for num, key in self.dialogue.items():
                if ".N" not in str(num):
                    print(f"\n{num}. {key[0]}")
            text = input("What do you want to ask? Type a number. Pres 'q' to quit.\n")
            if (text == 'q'):
                return 
            print(self.getDialogue(text))
            self.remDialogue(text)
        return
    @staticmethod
    def moveAll(npcs):
        for npc in npcs:
            npc.moveInPath(npc)
class Player:
    def __init__(self):
        self.inventory = []
        self.combinations = []
        self.understandings = {}
    def addUnderstanding(self, text, word):
        self.understandings[text] = word
    def getUnderstanding(self, text):
        return self.understandings[text]
    def addToInventory(self, thing):
        self.inventory.append(thing)
    def removeFromInventory(self, thing):
        self.inventory.remove(thing)
    def getInventory(self):
        """Returns the player's inventory in the form of a numerated list."""
        things = ""
        if(len(self.inventory) == 0):
            return "\n You have nothing."
        if len(self.inventory)>0:
            things += "\nYou have: "
            for num, x in enumerate(self.inventory):
                things += f"\n{num}. " + x.name
            return things
        else:
            return ""
    def saveGame(self, text):
        """Creates a file with all your inputs up to that point."""
        file = input("What would you like to name your save file?")
        if(os.path.exists(f"{file}.txt")):
            os.remove(f"{file}.txt")
        with open(f"{file}.txt", "w") as file:
            for pos, line in enumerate(text):
                if pos < len(text)-1:
                    file.write(line + "\n")
                else:
                    file.write(line)
    def useSave(self):
        """Plays the save file as if the user input the commands."""
        file = input("What is the name of your save file?")
        with open(f"{file}.txt", "r") as file:
            while True:
                line = file.readline().strip()
                if not line and "save game" not in line:
                    return
                yield line
                
                    
    def addCombination(self,thing, toCombine, output, oneWay=False):
        self.combinations.append([thing, toCombine, output])
        if(oneWay == False):
            self.combinations.append([toCombine, thing, output])
    def getCombination(self, thing, toCombine):
        """Returns the combination and removes the items involved."""
        for combination in self.combinations:
            if thing in combination and toCombine in combination:
                if thing in self.inventory and toCombine in self.inventory:
                    self.inventory.remove(thing)
                    self.inventory.remove(toCombine)
                    self.inventory.append(combination[2])
                    return f"\nYou've gotten: {combination[2].name}"
            return "That didn't work"
class Thing:
    def __init__(self, name, description, **kwargs):
        self.name = name
        self.description = description
        self.use = ""
        self.things = []
        self.properties = []
        self.interactions = []
        self.portable = True
        self.transparent = True
        self.listed = True
        #changes properties if listed in arguments, otherwise creates new property
        for item in kwargs:
            if("portable" in item):    
                self.portable = kwargs["portable"]
            elif("transparent" in item):   
                self.transparent = kwargs["transparent"]     
            elif("listed" in item):   
                self.listed = kwargs["listed"]
            else:
                self.properties.append(item)
    def addDefUse(self, use):
        self.use = use
    def getDefUse(self):
        return "\n"+self.use
    def addInteraction(self, word, text):
        """adds an additional parser action that doesn't do anything."""
        self.interactions.append([word, text])
    def getInteraction(self, word):
        """gets an additional parser action from the list of added actions."""
        for interaction in self.interactions:
            if interaction[0] == word:
                return interaction[1]
        return ""
    def addThing(self, thing):
        self.things.append(thing)
    def removeThing(self, thing):
        self.things.remove(thing)
    def getThings(self) -> str:
        """Lists everything listable within this thing."""
        listed = 0
        unlisted = 0
        
        for thing in self.things:
            if thing.listed == True:
                listed += 1
            else:
                unlisted +=1
        if listed == 0 and unlisted > 0:
            return ""
        """Returns a string containing a list of items contained within an object."""
        string = f"\nThere are these things around {self.name}: "
        if(len(self.things) > 0):   
            for x in self.things:
                string += x.name + " "
            return string
        else:
            return ""
    

class Room:
    def __init__(self, name, firstDescription):
        self.name = name
        self.descriptions = {}
        self.descriptions[0] = firstDescription
        self.exits = []
        self.things = []
        self.witnessed = 0
        self.hint = "\nThere are no extra hints for this room."
        self.image = ""
        self.musicPath = ""
    def addMusic(self, fileName):
        fileName = resource_path(fileName)
        if(fileName.count("\\") > 1):
            self.musicPath = fileName
            return
        self.musicPath = fileName
    def playMusic(self):
        if os.path.isfile(self.musicPath):
            #checks if the file exists, then checks if something is already playing.
            if playback.playing:
                playback.stop()
                playback.load_file(self.musicPath)
                playback.set_volume(0.5)
                playback.loop_at_end(True)
                playback.play()
            else:
                playback.load_file(self.musicPath)
                playback.set_volume(0.5)
                playback.loop_at_end(True)
                playback.play()
    @staticmethod
    def revDirection(direction) -> str:
        """Takes a direction and reverses it, for use in a Room method. If you want more directions, add them and their inverse here."""
        source = {"north":"south",#regular cardinal directions
                  "east":"west",
                  "up":"down",#spacial directions
                  "in":"out",
                  "north-west":"south-east",#extra cardinal directions 
                  "north-east":"south-west",}
        if direction in source.keys():
            return source[direction]
        if direction in source.values():
            for key, value in source.items():
                if direction == value:
                    return key
        
        
    def addAdjoiningRoom(self, room, direction, oneWay=False):
        """Adds a room by specifying where you come from, where you're going, and in what direction.
        Additionally, you can specify the oneWay argument to be true in order to get a 1-way door."""
        self.exits.append([room, direction])
        if oneWay == False:
            room.exits.append([self, Room.revDirection(direction)])
    def addThing(self, thing):
        """Adds the object to the list of objects in the room."""
        self.things.append(thing)
    def removeThing(self, thing):
        """Removes the object from the list of objects in the room."""
        self.things.remove(thing)
    
    def getThings(self):
        """Returns a list of names of the objects in the location. If there are none, it returns an empty string."""
        things_exist = True in map(lambda thing: thing.listed, self.things)
            #checks if there is something listed
        names = ""
        if(things_exist):
            """Checks if there is an npc or a thing which are listed"""
            for thing in self.things:
                if(thing.listed == True):
                    names += " " + thing.name + ", "
            return "\nYou see: " + names
        else:
            return ""
    def incrementWitnessed(self):
        self.witnessed += 1
    def addDescription(self, witnessed, description):
        """Adds a description for a given value or values of times having visited a room, or if you've visited a room more than n times."""
        if (type(witnessed) == type("text")):
            if("-" in witnessed):
                #when adding a range of descriptions
                nums = witnessed.split("-")
                a = int(nums[0])
                b = int(nums[1])
                for num in range(a,b+1):
                    self.descriptions[num]=description
                self.special = False
                return
        self.descriptions[witnessed]=description
    def getDescription(self):
        """Returns the specific description asked for in the number argument."""
        #how do I make it return the elongated description until the number has been matched?
        #finds your additional room description, if it exists.
        number = self.witnessed
        for num, desc in self.descriptions.items():
            if num == number:
                return desc
        #final case, returns the default room description
        return self.descriptions[0]
    def changeDescription(self, number, newDescription):
        for num in self.descriptions.keys():
            if num == number:
                self.descriptions[number] = newDescription
                return 
        raise Exception("You input an invalid description identifier.")
    def setHint(self, hint):
        self.hint = hint
    def getHint(self):
        return "\n"+self.hint
    def addImage(self, fileName):
        fileName = resource_path(fileName)
        if(fileName.count("\\") > 1):
            self.image = climage.convert(fileName, is_unicode=True)
            return
        self.image = climage.convert(fileName, is_unicode=True)          
    def getImage(self):
        if self.image != "":
            return self.image
        return ""
    
    
    
class BigRoom(Room):
    """A class which generates an x by y grid of Rooms. It might be good for blind mazes or long corridors, I suppose. You could also make REALLY big rooms."""
    def __init__(self, dimensions, name, firstDescription):
        super().__init__(name, firstDescription)
        self.x = int(dimensions.split("x")[0])
        self.y = int(dimensions.split("x")[1])
        self.rooms = {}
        for pos in range(self.x*self.y):
            #creating rooms
            self.rooms[pos] = Room(f"{name}{(pos%self.x)+1}x{(pos//self.x)+1}", firstDescription)

        if self.y > 1 and self.x > 1:
            #block connecting mode.
            for y in range(self.y):
                #connecting horizontally
                for num in range(self.x-1):
                    self.rooms[num+self.x*y].addAdjoiningRoom(self.rooms[num+1+self.x*y], "east")
            for x in range(self.x):
                #connecting vertically
                for depth in range(self.y-1):
                    self.rooms[x + self.x*depth].addAdjoiningRoom(self.rooms[x + self.x*(depth+1)], "south")
            for x in range(self.x-1):
                #connecting top left to bottom right diagonally
                for depth in range(self.y-1):
                    self.rooms[x+self.x*depth].addAdjoiningRoom(self.rooms[x+1+self.x*(depth+1)], "south-east")
            for x in range(1,self.x):
                #connecting top right to bottom left diagonally
                for depth in range(self.y-1):
                    self.rooms[x+self.x*depth].addAdjoiningRoom(self.rooms[x-1+self.x*(depth+1)], "south-west")
        else:
            #corridor connecting mode 
            if self.y == 1:
                for num in range(len(self.rooms)-1):
                        self.rooms[num].addAdjoiningRoom(self.rooms[num+1], "east")
            if self.x == 1:
                for num in range(len(self.rooms)-1):
                        self.rooms[num].addAdjoiningRoom(self.rooms[num+1], "south")
    def addDescription(self, origin, witnessed, description):
        if type(origin) == type("beep") and "x" in origin:
            x = int(origin.split("x")[0])-1
            y = int(origin.split("x")[1])-1
            self.rooms[x+self.x*y].addDescription(witnessed, description)
            return
        else:
            self.rooms[origin-1].addDescription(witnessed, description)
    def addDescriptionBig(self, witnessed, description):
        for room in self.rooms.values():
            room.addDescription(witnessed, description)
    def getThings(self):
        names =""
        for pos in self.x*self.y-1:
            things_exist = True in map(lambda thing: thing.listed, self.rooms[pos].things)
            if(things_exist):
                """Checks if there is an npc or a thing which are listed"""
                for thing in self.things:
                    if(thing.listed == True):
                        names += " " + thing.name + ", "
        if name != "":
            return "You see: "+names
        else:
            return ""
    def addAdjoiningRoom(self, origin, destination, direction, oneWay=False):
        """1 is the top left, and it goes to the right, then down a column."""
        if "x" in origin:
            x = int(origin.split("x")[0])-1
            y = int(origin.split("x")[1])-1
            self.rooms[x+self.x*y].addAdjoiningRoom(destination, direction, False)
            if oneWay:
                destination.addAdjoiningRoom(x+self.x*y, direction, False)
            return
        self.rooms[origin-1].addAdjoiningRoom(destination, direction, False)
        if oneWay:
            destination.addAdjoiningRoom(origin, direction, False)
def randomWord():
    random_list = ['me', 'VIII', 'tried', 'slavers', 'to', 'deal?', 'cells,', 'blindness,', 'ride', 'drag', 'rounds', 
    'All', 'hide,', '239', 'lap,', 'Who', 'sleep,', 'Remaining', 'nothing', 'the', 'bilious', 'Food', 
    'stand', 'want.', 'bosom', 'books,', 'run', 'things', 'destroy', 'within', 'yesterday', 'history,',
    'enjoy', 'mountain', 'grew,', 'Into', 'like', 'so!', 'take', 'Peaceful', 'enough?', 'airwaves,', 'Tax', 
    'dominance', 'crime,', 'symbol?!', 'waste', 'eat,', 'Thereâ€™s', 'showing', 'Tool', 'living', 'drugs', 'internal', 
    'meat', 'deranged', 'Subatomic', 'owned', 'found,', 'meatloaf', 'another', 'Wake', 'actions', 'earthworm', 'breath',
    'Harmful', 'burden,', 'fingers', 'mine,', 'kicks', 'moon', 'Akin', 'Are', 'beautiful', 'front,', 'undiscovered', 'soundly',
    'know', 'sun', 'Thunderstorm', 'conflict', 'utmost', 'sentence,', 'lack', 'climbing,', 'courage!', 'adventure,', 'death,',
    'Compost,', 'lights.', 'plants', 'other', 'diers,', 'sky', 'over.', 'feel', 'aâ€¦', 'Smack', 'maidens', 'make', 'set', 'listens', 
    'loss,', 'situation,', 'Movie,', 'That', 'way!', 'which', '3.', 'solving', 'birthplace', 'dipped', 'peel', 'sit', 'rather', 'watchful,', 
    'seeds', 'bitter', 'it', 'anther', 'in,', 'lecherous,', 'wake', 'nada!', 'must', 'raw,', 'namely', 'Danced', 'Need.', 'Iâ€™d', 'ruminations',
    'considering', 'Center,', 'gone,', 'glistens', 'gallantry,', 'director,', 'consequence.', 'cannot', 'decay,', 'prison,', 'no', 'chew,', 'tame',
    'nucleotides,', 'swear', 'with', 'all', 'dawn', 'aching,', 'she', 'duties', 'center', '241', 'flowers', 'annoyredom', 'grows', '1.', 'while', 'mental',
    'well.', 'yang', 'not,', 'astray,', '4.', 'blush,', 'transplant,', 'lying,', 'ordeal,', 'yearly', 'dinosaur,', 'enough', 'Stomping', 'improvement,',
    'little', 'upon', 'Program,', 'through,', 'whom', 'scorched', 'scorpion', 'journey,', 'hunt', 'white', 'Human,', 'mock,', 'Sand', 'products', 'joy,', 
    'plant', 'come,', 'shining', 'Of', 'Flying', 'reason', 'mate', 'adventurer,', 'first', 'chariot', 'monkey', 'stew,', 'powder.', 'hospital,', 'external', 
    'toothed', 'oh', 'extinct,', 'ancient,', 'dropped', 'quite', 'brow', 'Word,', 'repeat', 'rain.', 'chest', 'our', 'told', 'While', 'Am', 'let', 'itâ€™s', 
    'myself.', 'detritivores,', 'watch', 'live?!', 'languidly,', 'heart?', 'organic', 'moment,', 'full', 'dragon', 'comes', 'how', 'yin', 'on', 'Grand', 
    'criminal,', 'Perhaps', 'Dawning', 'row', 'you', 'problem', 'be,', 'have', 'expired', 'elephant', 'much', 'maiden,', 'Illness,', 'would', 'sanity', 
    'clip', 'being', 'colour,', 'become', 'Gray', 'claim', 'yurt', 'kind.', 'own', 'lacking', 'chess', 'lion,', 'me,', 'inside,', '5.', 'keep', 'tap,', 
    'Victories', 'priest,', 'prideful', 'stare,', 'growth,', 'defeat,', 'Before', 'schedule,', 'celebrations.', 'waiting', 'eritrea', 'want', 'Racked',
    'sensation', 'grant', 'do', 'brought', 'Earth', 'truth', 'Drowsy', 'impersonation', 'yesteryear', 'haunt', 'is', 'nucleolus,', 'form', 'work,', 'nail',
    'whose', 'clop', 'Walking', 'Clap', 'stress,', 'reproduction?', 'colour', 'spirit', 'Friend', 'dromedary', 'job,', 'apart?', 'wonder,', 'her', 'night',
    'one,', 'destiny', 'Watch', 'destined', 'ew', 'Atoms,', 'death.', 'energy,', 'cracked', 'before', 'hate', 'fire,', 'dog,', 'party', 'Such', 'damn', 
    'Strength', 'world', 'ask', 'longer,', 'knife', 'exemplary', 'giants', 'kerfluffled', 'nano', 'reverie.', 'mirror,', 'Fighting', 'Men,', 'seek,', 
    'close', 'shame,', 'it.', 'No', 'hagness', 'wham', 'online', 'inkling', 'your', 'reality', 'dent', 'cat,', 'retribution?', 'Petals', 'lions', 'air,',
    'doth', 'films,', 'track', 'bringing', 'For', 'claws', 'tell', 'isnâ€™t', 'share,', 'wonâ€™t', 'Rats', 'change', 'open,', 'Wild', 'demented',
    'medicine,', 'golden', 'drown', 'armour', 'Fear', 'wordplay', 'children,', 'Upon', '242', 'many?', 'thrown', 'Elated', 'here,', 'am',
    'Enjoy', 'toil', 'molding', 'drive', 'peak', 'Now', 'code,', 'Come', 'house', 'foreclosed,', 'bit,', 'this,', 'thinker,', 'rejection,',
    'us', 'ape,', 'courage,', 'tree', 'flew,', 'that', 'Ruinous', 'an', 'manish', 'Locked', 'Money,', 'hands', 'scissors,', 'instinct,', 
    'Assassin,', 'transition', 'not', 'driving', 'drop', 'woman', 'ground', 'inner', 'failure,', 'beyond', 'him', 'sounds', 'gray', 'side',
    'room', 'distasteful,', 'face', 'Lost,', 'tram', 'coming,', 'Itâ€™s', 'had', 'morae,', 'Sluttifying,', 'Knight', 'legs', 'Earring', 'some',
    'frail', 'foul,', 'drop,', 'bed,', 'Besides,', 'live', 'tells', '235', 'Everyday', 'do,', 'grey', 'see.', 'has', 'need', 'be', 'programmer',
    'timely', 'country,', 'you', 'Embarrassment', 'dodged', 'light', 'Yak', 'thinkingâ€™s', 'plan,', 'timid,', 'warmth,', 'Travels,', 'dam', 'by']
    return random.choice(random_list)
def parser(room, user_input, player, rooms=[], things=[], npcs=[]):
    text = user_input
    NPC.moveAll(npcs)
    container = {"n":"north","s":"south","e":"east","w":"west",
                 "out":"out","in":"in",
                 "nw":"north-west","ne":"north-east","sw":"south-west","se":"south-east"}
    if(type(user_input) == type(["apple"])):
        user_input = user_input[-1]
    """Takes user input and converts it into 2 keywords that then are matched against possible commands."""
    if(len(user_input.split()) == 1):
        #a single word case.
        user_input = user_input.lower().strip()
        if(user_input in container.keys()):
            #trying to move in a direction using a shorthand of the direction
            for exit in room.exits:
                if(container[user_input] == exit[1]):
                    exit[0].incrementWitnessed()
                    return exit[0], exit[0].getDescription() + exit[0].getThings()
        if(user_input == "i" or user_input == "inventory"):
            return room, player.getInventory()
        
        if(user_input == "help"):
            return room, """You can MOVE/GO 'LOCATION', LOOK 'AROUND'/'OBJECT'/'ITEMS', GET 'ITEM' or USE 'ITEM' or USE 'item' ON 'another item',
                            and can 'QUIT GAME' If you REALLY need a hint, then GET HINT should help. 
                            \nThe directions are mostly cardinal (those on a compass).
                            In addition to the directions, you could use just the first letter, or first 2 letters for extra cardinal directions.
                            You could also type i or inventory to view your inventory. You can save with SAVE"""
        if(user_input == "save"):
            player.saveGame(text[:-1])
            return room, "\nYou have saved the game. To play this save, exit then return and run USE SAVE."
    if len(user_input.split()) == 2:
        #a double word case.
        first, second = user_input.lower().strip().split()
        second_full = user_input.lower().strip().split()[1:]
        
        
        if(first not in ["use","look", "get", "help", "move", "talk", "quit"]):
            """Code which returns the additional interactions."""
            for item in player.inventory:
                if item.name == second:
                    return room, item.getInteraction(first)
            for item in room.things:
                if item.name == second:
                    return room, item.getInteraction(first)
        #room specific code and use code--------------------------------------------------------------------------
        #ideas: teleporation, keys + use, using items together, locked doors, being in a certain room changing all rooms.
        
        
        
        
        if(first == "use"):
            #specific object/room interactions-------------------------------------------
            #for example, checking if you're in a specific room before doing something.
            if(area.name == "vendor for bananas" and coconut in player.inventory and "coconut" ==second):
                print("Tim says: Thank you, I guess I now can clean that ball pit.")
                for thing in area.things:
                    if thing.name == "tim":
                        npc = thing
                area.things.remove(npc)
                player.inventory.remove(coconut)
                ball_pit.addAdjoiningRoom(secret, "north")
                ball_pit.addAdjoiningRoom(outside_land, "west", oneWay=True)
                return room, room.getDescription() + room.getThings()
            if(lighter in player.inventory and "lighter" == second):
                print("You lit a light")
                basement.addDescriptionBig(0, "It is dark and damp down here, it's like 9 rooms in one.")
                basement.addDescription("2x2", 0, "It is dark and damp down here, the way up is here.")
                basement.addDescription("1x2", 0, "It is dark and damp down here, you can go west here")
                basement.addDescription("2x1", 0, "It is dark and damp down here, you can go north here.")
                basement.rooms[1].addAdjoiningRoom(murder_room, "north")
                basement.rooms[3].addAdjoiningRoom(clown_corner, "west")
                return room, room.getDescription() + room.getThings()
                
            #end of specific object/room interactions------------------------------------
            for x in player.inventory:
                #display default use
                if x.name == second:
                    return room, x.getDefUse()
            for x in room.things:
                if x.name == second:
                    return room, x.getDefUse()
            if(second == "save"):
                player.useSave()
            return room, "You can't do that here with that."
        #room specific and use code------------------------------------------------------------------------------
        
        
        if(first == "look"):
            """Checks if your first choice was to look at something, then finds the correct text to output for what you typed."""
            if(second == "around" or second == "room"):
                return room, room.getDescription() + room.getThings()
            elif (second == "inventory" or second == "items"):
                return room, player.getInventory()
            else:
                #if the object is in the player's inventory
                if second in player.getInventory():
                      for x in player.inventory:
                        if x.name.lower() == second:
                            return room, x.description

                #if the object isn't in the player's inventory.
                else:
                    for thing in room.things:
                        if thing.name.lower() == second :
                            if thing.portable:
                                #if you can pick up the object 
                                return room, thing.description + thing.getThings()
                            else:
                                #if you can't pick up the object
                                return room, thing.description + thing.getThings()
                    return room, f"{second} isn't here."

        if(first == "get"):
            """Manages the user acquiring items, and removing that same item from the room it was in to put it in the inventory."""
            if second == "hint":
                return room, room.getHint()
            for x in room.things:
            #gets an item if it isn't in another item and you can pick it up.
                if second == x.name:
                    #if you can pick up the item
                    if(x.portable):
                        player.addToInventory(x)
                        room.removeThing(x)
                        return room, "\nYou got it."
                    else:
                        return room, "\nYou can't pick that up."
            for thing in room.things:
            #gets an item if it is in another item and you can pick it up.   
                for x in thing.things:
                    if second == x.name:
                        #if you can pick up the item
                        if(x.portable):
                            player.addToInventory(x)
                            thing.removeThing(x)
                            return room, "\nYou got it."
                        else:
                            return room, "\nYou can't pick that up."
            return room, "That's either not here or you can't pick that up."
            

        if(first == "help" and second == "me"):
            """Checks if your choice was to get help, which then returns some advice on playing the game."""
            return room, """You can MOVE/GO 'LOCATION', LOOK 'AROUND'/'OBJECT'/'ITEMS', GET 'ITEM' or USE 'ITEM' or USE 'item' ON 'another item',
                            and can 'QUIT GAME' If you REALLY need a hint, then GET HINT should help. 
                            \nThe directions are mostly cardinal (those on a compass).
                            In addition to the directions, you could use just the first letter, or first 2 letters for extra cardinal directions.
                            You could also type i or inventory to view your inventory. You can save with SAVE"""

        if(first == "move" or first == "go"):
            """Checks if your choice was to move, then it looks through all the exits of the room
            to see if there is a matching exit to your move."""
            for exit in room.exits:
                if(second == exit[1]):
                    exit[0].incrementWitnessed()
                    return exit[0], exit[0].getDescription() + exit[0].getThings()
                if(second in container.keys() and container[second] == exit[1]):
                    exit[0].incrementWitnessed()
                    return exit[0], exit[0].getDescription() + exit[0].getThings()
            else:
                return room, "\n You can't move in that direction here."
               
        if(first == "quit" and second == "game"):
            return room, "quit"
        return room, "You input an invalid command. Try 'look'ing or 'move'ing"       
    
    
    elif (len(user_input.split()) > 2):
        """If the length of the input is greater than 2"""
        for key, word in player.understandings.items():
            """Allows for additional understandings to be used, then converted into other forms"""
            if(key in user_input):
                return parser(room, player.getUnderstanding(user_input), player, rooms, things)
        words = user_input.lower().strip().split()
               
        if(words[0] == "use"):
        #specific object/room interactions-------------------------------------------
        #for example, checking if you're in a specific room before doing something.
            if(area.name == "pirate cove" and pirate_hat in player.inventory and words[1:] == "pirate hat".split()):
                area.changeDescription(0, "After putting on the pirate hat, you are now accepted as one of the pirates.")
                for thing in room.things:
                    thing.listed = True
                area.addAdjoiningRoom(back_of_vendor_coconuts, "north-east")
                return room, room.getDescription() + room.getThings()
        #end of specific object/room interactions------------------------------------
            for x in player.inventory:
                #display default use
                if x.name.split() == words[1:]:
                    return room, x.getDefUse()
            for x in room.things:
                if x.name.split() == words[1:]:
                    return room, x.getDefUse()
        if (words[0] == "use" and words[2] == "on"):
            "Allows for the combining of items."
            thing1 = words[1:words.find("on")]
            thing2 = words[3:]
            for thing in player.inventory:
                if thing1 == thing.names.split():
                    thing1 = thing
                if thing2 == thing.name.split():
                    thing2 = thing
            if type(thing1) == type("String") or type(thing2) == type("String"):
                return room, "Those things aren't in your inventory"
            else:
                return room, player.getCombination(thing1, thing2)
        if(words[0] == "look" and words[1] == "at"):
            for thing in player.inventory:
                if thing.name.split() == words[2:]:
                    return room, thing.description + thing.getThings()
            for thing in room.things:
                if thing.name.split() == words[2:]:
                    return room, thing.description + thing.getThings()
            if(words[2] == room.name):
                return room, room.getDescription() + room.getThings()
            
        if(words[0] == "get"):
            for x in room.things:
            #gets an item if it isn't in another item and you can pick it up.
                if words[1:] == x.name.split():
                    #if you can pick up the item
                    if(x.portable):
                        player.addToInventory(x)
                        room.removeThing(x)
                        return room, "\nYou got it."
                    else:
                        return room, "\nYou can't pick that up."
            for thing in room.things:
            #gets an item if it is in another item and you can pick it up.   
                for x in thing.things:
                    if words[1:] == x.name.split():
                        #if you can pick up the item
                        if(x.portable):
                            player.addToInventory(x)
                            thing.removeThing(x)
                            return room, "\nYou got it."
                        else:
                            return room, "\nYou can't pick that up."
            
            
            
            
            
        if(words[0] == "talk" and words[1] == "to"):
            for npc in room.things:
                if words[2:] == npc.name.split():
                    npc.talk()
                    return room, room.getDescription() + room.getThings()
        return room, "\nI didn't understand that."
    else:
        return room, "\nI didn't understand that."
def resource_path(relative_path, forPyinstaller = True):
    """A function I stole to be able to run smaller executables"""
    if(not forPyinstaller):
        return relative_path
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)
player = Player()
playback = Playback()
#"code"-------------------------------------------------------------------------------------- 
#You can write room specific or use interactions code in the parser function
#
#ROOMS----
#The place you start in must always be called "start"
#add rooms with: [room] = Room("room", "description")
#join rooms with: [room].addAdjoiningRoom([destination], "direction_text") add a oneWay=True or just "True" to the end to make it a one way.
#add new description when seen n times: [room].addDescription(n, "description"), n can be a range "a-b" change the default room by setting n to 0.
#add music for the room with: [room].addMusic(fileName)
#add image for the room with: [room].addImage(fileName)
#BIGROOMS-----
#add a BigRoom with: [bigRoom] = BigRoom("name", "description", width x height)
#add adjoining rooms with: [bigRoom].addAdjoiningRoom(roomNumber, destination, direction) add a oneWay=True or just "True" to make it a one way.
#joining a room to a BigRoom should be done from the BigRoom.
#change room description in BigRoom: [BigRoom].addDescription("widthxheight"/pos, witnessed, description) make witnessed 0 to make it default.
#to access specific rooms within BigRoom: [BigRoom].rooms[pos]
#OBJECTS----
#add object: [object] = Thing("object", "description", portableState(defaults to true), transparentState(defaults to true), listed(default is true))
#join object to room: [room].addThing([object])
#add use attempt text: [object].addDefUse("text for use")
#add interaction text: [object].addInteraction("action", "text to display")
#add object to object (like a table containing things) [object].addThing([otherObject])
#HINTS----
#You can do [room].setHint("text") to add a hint that could be used.
#PLAYER----
#you could add the ability to merge items through [player].addCombination([item1], [item2], [output]) it automatically adds the other direction, unless you add True or oneWay=True
#You can add an understanding of a certain bunch of text as a set of simpler commands, with [player].addUnderstanding(textToUnderstand, newText)
#NPCs----
#you can create NPCs with [npc] = NPC("name", "description", [startingLocation]"
#you can give dialogue with: [name].addDialogue(num, question, answer)
#you can give NPCs pathing with: [name].setupPath("path list")
#join NPC to room: [room].addThing([NPC])
#---------------------------------------------------------------------------------------------
openingText = "Welcome to Pirate Lagoon! I'd advice you move around and check your list at least twice.\nif you need help, which you probably do, just type 'help me' or 'help'\n\n\n"
start = Room("Ocean", "Your pirate mates dropped you off here. You are wading in water,\nand see an island east of you and one north of you. \nThey might be in swimming distance.")
player.addUnderstanding("swim north", "go north")
player.addUnderstanding("swim east", "go east")
regularly_shaped_island = Room("Regularly shaped island", "You floundered onto shore and walked inland to this location. You can go back west to the shore where you can swim again, or you can head north, south, or east.")
regularly_shaped_island.addImage("files/regularly_shaped_island.png")
oddly_shaped_island = Room("Oddly shaped island", "You climbed onto the rocks, seeing north a vendor, west a house, and east something colourful.") 
start.addAdjoiningRoom(regularly_shaped_island, "east", oneWay=True)
start.addAdjoiningRoom(oddly_shaped_island, "north")
start.addImage("files/ocean.png")
#regularly_shaped_island
hut = Room("hut", "You walk into the hut, seemingly abandoned. You're surprised it's still standing. Not primitive after all.")
table = Thing("table", "It's a table, wooden, probably rotting a bit.", portable=False)
furniture = Thing("furniture", "It's a bunch of old, moldy bread, stewing, green furniture.", portable=False)
lighter = Thing("lighter", "A tiny flamethrower.")
table.addThing(lighter)
hut.addThing(table)
hut.addThing(furniture)

vendor_coconuts = Room("vendor for coconuts", "You walk into the store, surprised at how modern it seems, besides it only selling coconuts.")
pirate_cove = Room("pirate cove", "You walk around the outskirts, hearing myriad quiet conversation in accents oddly similar to the west country accent.")
haggard_pirate = NPC("haggard pirate", "He smells like booze, and emits an aura of being quite dangerous.", listed=False)
haggard_pirate.addDialogue(1, "Do you know what there is to eat around here?", "There's a backdoor north-east which leads to the coconut guy's product. I have no idea why it is that way.")
haggard_pirate.replaceDialogue(1, 1, "What's that direction again?", "North-east, you scatterbrain!", persistent=True)
haggard_pirate.addDialogue(2, "Who are you?", "Does it matter?")
haggard_pirate.addDialogue(3, "What's it like being a pirate?", "Aren't you a pirate too??")
pirate_cove.addThing(haggard_pirate)
    #script for pirate hat
shore = Room("shore", "You can see the ocean from here quite well, if you so choose you could probably swim back there and try to find other lands.")
regularly_shaped_island.addAdjoiningRoom(shore, "west")
regularly_shaped_island.addAdjoiningRoom(hut, "north")
regularly_shaped_island.addAdjoiningRoom(pirate_cove, "south")
regularly_shaped_island.addAdjoiningRoom(vendor_coconuts,"east")
shore.addAdjoiningRoom(start, "west", oneWay=True)

bill = NPC("bill", "He is in a chipper uniform, for whatever reason you don't know. He seems to be the one running the shop.")
bill.addDialogue(1, "Can I buy a coconut?", "Didn't you see the sign? No stragglers from ships allowed to be sold coconuts. It's a rule.", persistent=True)
bill.addDialogue(2, "Who are you?", "I'm Bill.")
bill.addDialogue(3, "What do you do?", "I work here, selling coconuts.")
bill.addDialogue(4, "What can I do for you?", "I need a banana, maybe the next island over has a banana vendor. I'm really craving a banana right about now.")
bill.replaceDialogue(1, 4, "What was I supposed to get you again?", "I need a banana. I really need a banana.", persistent=True)
bill.addDialogue(5, "Where are we?", "In the middle of nowhere, where else?")
vendor_coconuts.addThing(bill)
vendor_coconuts.addImage("files/bvendor.png")
    #add script for giving bill the banana
                     
back_of_vendor_coconuts = Room("Backroom of coconut vendor", "You probably shouldn't be in here, but it's what it's.")
coconut = Thing("coconut", "It's just a coconut")
back_of_vendor_coconuts.addThing(coconut)
                     
                     
#oddly_shaped_island
clown_cove = Room("clowncove", "A circus tent filled with rocks and evidence that this region was dammed or something to move the water. Is it really a cove?")
stewey = NPC("stewey", "He points at his throat and gestures seemingly that he can't speak.")
stewey.setupPath(["west", "north", "south", "west", "east", "north", "south", "east"], clown_cove)
stewey.addDialogue(1, "Please show me a word", f"He signs the word for {randomWord()}")
for x in range(1, 50):
    stewey.replaceDialogue(x, 1, "Please show me a word", f"He signs the word for {randomWord()}")
    #add random word generator
    #stewey gives you a coconut if you talked to the banana dealer.
stewey.addDialogue(2, "What are you doing?", "He mimes his 2 fingers moving, so he is just walking?")
clown_cove.addThing(stewey)
vendor_bananas = Room("vendor for bananas", "It is a shop like area, but it is in disrepair and looks lower class.")
tim = NPC("tim", "He is wearing a gold t-shirt, a bowler hat, and is noticeably obese.")
tim.addDialogue(1, "Can I buy a banana?", "Read the sign, you look like you fell off a ship.", persistent=True)
tim.addDialogue(2, "How are you fat here?", "Bananas are high in fats, at least I hope thats it, otherwise it's a condition.")
tim.addDialogue(3, "What can I do for you?", "You could get me a coconut, I'm not picky. Maybe a coconut would do the trick.")
tim.replaceDialogue(1, 3, "What am I supposed to get you again?", "a coconut.", persistent=True)
vendor_bananas.addThing(tim)
vendor_bananas.addImage("files/bvendor.png")
oddly_shaped_island.addAdjoiningRoom(clown_cove, "east")
oddly_shaped_island.addAdjoiningRoom(vendor_bananas, "north")
    #add dialogue for giving tim a flaming coconut.

house = Room("house", "It is a standard brick house. Maybe 2 stories, unless the roof is high or that's the attic. You can go north or down the stairs")
dining_room = Room("dining room", "A standard dining room. You are scared, maybe you see something that will help?")
basement = BigRoom("3x3", "basement", "It is dark and damp down here, you probably should have a light.")
spoon = Thing("spoon", "This spoon helps you feel better and less scared of that room.")
dining_room.addThing(spoon)

house.addAdjoiningRoom(dining_room, "north")
basement.addAdjoiningRoom("2x2", house, "up")
basement.addDescription("2x2", 0, "It is dark and damp down here, you probably should have a light. It's like 9 rooms in one. The way up is here.")
basement.addDescription("1x2", 0, "It is dark and damp down here, you probably should have a light. You feel a crease in the wall here, but don't know what it is.")
basement.addDescription("2x1", 0, "It is dark and damp down here, you probably should have a light. You feel a crease in the wall here, but don't know what it is.")
pirate_hat = Thing("pirate hat", "It looks like you could dress like a pirate with this.")
basement.rooms[0].addThing(pirate_hat)
oddly_shaped_island.addAdjoiningRoom(clown_cove, "east")
oddly_shaped_island.addAdjoiningRoom(vendor_bananas, "north")
oddly_shaped_island.addAdjoiningRoom(house, "west")
#basement
    #rooms for the basement are only unlocked when you have a light here.
    #attach murder_room and clown_corner when there is a light.
murder_room = Room("murder room", "You see implements of torture and signs of struggle, oh so many signs of struggling. The walls are stained with blood. You can go south.")
murder_table = Thing("murder table", "It is covered in tools of slaughter", portable=False)
blood_stains = Thing("blood stains", "They are copious", portable=False)
murder_room.addThing(murder_table)
murder_room.addThing(blood_stains)

#clown corner
clown_corner = Room("clown corner", "There are signs of clowns having been here. You are afraid. You can go back east or you could go west down a slide.")
    #make tim clear the ballpit? Then add the secret and outside_land
    #make stewey show you a magic gesture?
ball_pit = Room("ball pit", "The unholy vapors are scorching your nostrils, you quickly search for a way out. There are stairs back up, and there is a door west of you.")
secret = Room("secret", "You found the easter egg, you win!")
outside_land = Room("outside land", "You made it outside of this hellscape.")
clown_corner.addAdjoiningRoom(ball_pit, "west")
    #make it so you need tim to have the flaming coconut first before getting outside.
#end of "code" ------------------------------------------------------------------------------



#Runs the game
output = "" 
#edit this print if you want a unique start text.
print(openingText)
area = start
moves = 0
text = []
try:
    print(area.getImage())
    area.playMusic()
    print(area.name.upper() + f", MOVES: {moves}" + "\n" + area.getDescription(), area.getThings())
    while output != "quit":
        text.append(input("-What do you want to do?"))
        if(text[0] == "use save"):
            for line in player.useSave():
                print(line)
                if line:
                    text.append(line)
                    area, output = parser(area, line.strip().lower(), player, rooms=[back_of_vendor_coconuts, basement, clown_corner, murder_room, ball_pit, secret, outside_land],things=[pirate_hat, coconut, lighter], npcs=[stewey])
                    moves+=1
                    print(area.getImage())
                    area.playMusic()
                    print(area.name.upper() + f", MOVES: {moves}"+ "\n" + output.strip())
            text[0] = ""
            text.remove("")
            continue
        print("\n\n")
        area, output = parser(area, text, player, rooms=[back_of_vendor_coconuts, basement, clown_corner, murder_room, ball_pit, secret, outside_land], things=[pirate_hat, coconut, lighter], npcs=[stewey])
        moves+=1
        print(area.getImage())
        area.playMusic()
        print(area.name.upper() + f", MOVES: {moves}"+ "\n" + output.strip())
except KeyboardInterrupt:
    if playback.active:
        playback.stop()
        

Welcome to Pirate Lagoon! I'd advice you move around and check your list at least twice.
if you need help, which you probably do, just type 'help me' or 'help'



▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄

-What do you want to do? move east





▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄

-What do you want to do? move west






SHORE, MOVES: 2
You can see the ocean from here quite well, if you so choose you could probably swim back there and try to find other lands.


-What do you want to do? move west





▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄